# Day 5 — Scikit-learn Pipelines & Tuned Mini-Project

Building a leak-free end-to-end Pipeline on the Titanic dataset using a
ColumnTransformer for mixed numeric and categorical data, incorporating
an engineered Title feature, and tuning the complete workflow with
5-fold GridSearchCV.

The final tuned pipeline is evaluated once on a held-out test set and
compared against an untuned baseline.

## Loading and Initial Inspection
Load the dataset and review its shape, structure, missing values, and descriptive statistics before building anything.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv("Titanic-Dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


The dataset has 891 rows and 12 columns, covering each passenger's class, name, sex, age, family aboard, fare, and survival outcome.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


.info() shows Age, Cabin, and Embarked have fewer non-null entries than the rest — confirmed exactly next.

In [4]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

.isnull().sum() confirms the gaps: Age missing 177, Cabin missing 687 (the large majority), Embarked missing 2.

In [5]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


.describe() gives the spread of the numeric columns, useful context before scaling and encoding decisions below.

## Engineering a Title Feature

Extract the passenger's title (e.g., Mr, Miss, Mrs, Master) from the
Name column and group rare titles into a single Rare category.

The engineered Title feature is treated as a categorical feature and is
included in the preprocessing pipeline for the final model.

In [6]:
df["Title"] = df["Name"].str.extract(
    r",\s*([^.]*)\.",
    expand=False
)

rare_titles = [
    "Lady", "Countess", "Capt", "Col", "Don",
    "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"
]

df["Title"] = df["Title"].replace(rare_titles, "Rare")

df["Title"].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Rare             22
Mlle              2
Mme               1
Ms                1
the Countess      1
Name: count, dtype: int64

Extracting the text between the comma and the period in Name creates the
Title feature. Common titles such as Mr, Miss, Mrs, and Master
appear frequently, while less common titles are grouped into a single
Rare category.

Grouping rare titles reduces the number of low-frequency categories while
preserving potentially useful information from the passenger's title.

In [7]:
X= df.drop(["Survived", "PassengerId" , "Cabin" , "Ticket" ,"Name"], axis =1)
y = df["Survived"]

X contains the predictor features after removing Survived (the target),
PassengerId, Cabin, Ticket, and the original Name column. The Title
feature extracted from Name is retained as a categorical feature.
y contains the target variable, Survived.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



An 80/20 train/test split (712/179 rows) is performed with random_state=42 and stratify=y to preserve the class balance.

## Step 1 — Building the Pipeline with a ColumnTransformer

Separate the features into numeric and categorical groups, including the
engineered Title feature. A ColumnTransformer will apply the appropriate
preprocessing to each group, including missing-value imputation, scaling for
numeric features, and one-hot encoding for categorical features.

In [9]:
numeric_cols = [
    "Age",
    "SibSp",
    "Parch",
    "Fare"
]

categorical_cols = [
    "Pclass",
    "Sex",
    "Embarked",
    "Title"
]

numeric_cols (Age, SibSp, Parch, Fare) and categorical_cols (Pclass, Sex, Embarked, Title) are separated so each group can be preprocessed appropriately. The engineered Title feature is included as a categorical feature and will be processed as part of the pipeline.

In [10]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

Two preprocessing pipelines are defined for the two feature groups. Numeric
features use median imputation followed by standardization, while categorical
features use most-frequent imputation followed by one-hot encoding. These
preprocessing steps will be integrated into the ColumnTransformer to ensure
that missing values and feature transformations are handled consistently
within the main Pipeline.

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

The ColumnTransformer combines the numeric and categorical preprocessing
pipelines into a single preprocessing step. Numeric features are imputed
using the median and then standardized, while categorical features are
imputed using the most frequent value and then one-hot encoded. This allows
both feature types to be processed consistently within the main Pipeline.

In [12]:
model = RandomForestClassifier(
    random_state=42
)

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])



The full Pipeline chains the preprocessing steps with a RandomForestClassifier
in a single leak-free workflow. During fitting, the imputer, scaler, and
encoder are learned only from the training data. When predicting on the test
data, the same fitted transformations are applied without refitting them.

## Establishing an Untuned Baseline

Fit the complete pipeline using the default RandomForestClassifier settings
and evaluate its performance on the held-out test set. This baseline provides
a reference point for measuring the improvement achieved through
hyperparameter tuning.

In [13]:
pipe.fit(X_train, y_train)

y_pred_baseline = pipe.predict(X_test)

baseline_accuracy = accuracy_score(y_test, y_pred_baseline)
baseline_precision = precision_score(y_test, y_pred_baseline)
baseline_recall = recall_score(y_test, y_pred_baseline)
baseline_f1 = f1_score(y_test, y_pred_baseline)

print("Baseline Accuracy :", baseline_accuracy)
print("Baseline Precision:", baseline_precision)
print("Baseline Recall   :", baseline_recall)
print("Baseline F1       :", baseline_f1)

Baseline Accuracy : 0.8100558659217877
Baseline Precision: 0.7777777777777778
Baseline Recall   : 0.7101449275362319
Baseline F1       : 0.7424242424242424


**Baseline Results:** Accuracy = 0.810, Precision = 0.778, Recall = 0.710, and F1 = 0.742.

These results provide the reference performance for the complete pipeline,
including the engineered Title feature, with missing-value imputation and
default RandomForestClassifier hyperparameters.

## Step 3 — Tuning the Full Pipeline with GridSearchCV

Tune the RandomForestClassifier hyperparameters within the complete Pipeline
using 5-fold cross-validation and F1-score as the evaluation metric. The
double-underscore syntax is used to access parameters of the model step
inside the Pipeline.

In [14]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [5, 10]
}

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

Best Parameters: {'model__max_depth': 10, 'model__n_estimators': 100}
Best CV F1: 0.7501667911376649


**Best Parameters:** max_depth = 10, n_estimators = 100.

**Best CV F1:** 0.7502.

The best configuration was selected using 5-fold cross-validation on the
training data, with F1-score as the scoring metric.

In [15]:
best_model = grid.best_estimator_

y_pred_final = best_model.predict(X_test)

final_accuracy = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final)
final_recall = recall_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)

print("Final Accuracy :", final_accuracy)
print("Final Precision:", final_precision)
print("Final Recall   :", final_recall)
print("Final F1       :", final_f1)

Final Accuracy : 0.8268156424581006
Final Precision: 0.8166666666666667
Final Recall   : 0.7101449275362319
Final F1       : 0.7596899224806202


**Final Tuned Pipeline Results:** Accuracy = 0.827, Precision = 0.817,
Recall = 0.710, and F1 = 0.760.

Compared with the untuned baseline F1-score of 0.742, the final tuned
pipeline improves F1 to 0.760. Accuracy also increases from 0.810 to 0.827,
while precision improves from 0.778 to 0.817. Recall remains at 0.710.

The final model uses the best hyperparameters selected through 5-fold
GridSearchCV and is evaluated once on the held-out test set.

## Step 4 — Final Tuned Pipeline vs. Baseline

Evaluate the best pipeline selected by GridSearchCV once on the held-out test
set. Compare the final tuned model with the untuned baseline to measure the
effect of hyperparameter tuning on the model's performance.

In [16]:
print("Baseline F1 :", baseline_f1)
print("Final F1    :", final_f1)
print("Improvement  :", final_f1 - baseline_f1)

Baseline F1 : 0.7424242424242424
Final F1    : 0.7596899224806202
Improvement  : 0.01726568005637774


**Final Tuned Pipeline Results:** Accuracy = 0.827, Precision = 0.817,
Recall = 0.710, and F1 = 0.760.

Compared with the untuned baseline, the final tuned pipeline improves the
F1-score from 0.742 to 0.760, an improvement of approximately 1.73
percentage points. Accuracy increases from 0.810 to 0.827, while precision
increases from 0.778 to 0.817. Recall remains unchanged at 0.710.

The final result demonstrates the benefit of hyperparameter tuning while
maintaining a leak-free workflow that combines the engineered Title feature,
ColumnTransformer-based preprocessing, and GridSearchCV within a single
Pipeline.